# Clean up extracted conversations

In [1]:
input_dir = '../output/dialogues/pkna-*.json'

In [4]:
import glob

input_files = glob.glob(input_dir)
input_files.sort()

input_files[:10], len(input_files)

(['../output/dialogues/pkna-0-2-conv-005-0.json',
  '../output/dialogues/pkna-0-2-conv-005-1.json',
  '../output/dialogues/pkna-0-2-conv-005-2.json',
  '../output/dialogues/pkna-0-2-conv-005-3.json',
  '../output/dialogues/pkna-0-2-conv-011-0.json',
  '../output/dialogues/pkna-0-2-conv-011-1.json',
  '../output/dialogues/pkna-0-2-conv-011-2.json',
  '../output/dialogues/pkna-0-2-conv-011-3.json',
  '../output/dialogues/pkna-0-2-conv-011-4.json',
  '../output/dialogues/pkna-0-2-conv-011-5.json'],
 1046)

In [6]:
# Better sort the files by pkna, conv, and page
import re

def input_sort_key(filename: str) -> tuple:
    m = re.search(r'pkna-(.+)-conv-(.+)-(\d)+\.json', filename)
    if not m:
        raise ValueError(f"Filename {filename} does not match expected pattern.")
    pkna = float(m.group(1).replace('-', '.'))
    conv = int(m.group(2))
    page = int(m.group(3))
    return (pkna, conv, page)

input_files.sort(key=input_sort_key)
input_files[:10]

['../output/dialogues/pkna-0-conv-029-0.json',
 '../output/dialogues/pkna-0-conv-029-1.json',
 '../output/dialogues/pkna-0-conv-029-2.json',
 '../output/dialogues/pkna-0-conv-029-3.json',
 '../output/dialogues/pkna-0-conv-029-4.json',
 '../output/dialogues/pkna-0-conv-029-5.json',
 '../output/dialogues/pkna-0-conv-029-6.json',
 '../output/dialogues/pkna-0-conv-029-7.json',
 '../output/dialogues/pkna-0-conv-040-0.json',
 '../output/dialogues/pkna-0-conv-040-1.json']

In [18]:
import json
from dataclasses import dataclass

@dataclass
class Bubble:
    character: str
    text: str
    probability: int

@dataclass
class Take:
    dialogue: list[Bubble]

@dataclass
class Page:
    model: str
    prompt_version: str
    pkna: str
    page: str
    takes: list[Take]


def read_json_file(filename: str) -> dict:
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)

def dict_to_page(d: dict) -> Page:
    return Page(
        model=d['model'],
        prompt_version=d['prompt_version'],
        pkna=d['pkna'],
        page=d['page'],
        takes=[
            Take(dialogue=[
                Bubble(
                    character=b['character'],
                    text=b['text'],
                    probability=b['probability']
                )
                for b in t['dialogue']
            ])
            for t in d['takes']
        ]
    )

input_contents = [
    dict_to_page(read_json_file(filename))
    for filename in input_files
]

input_contents[:1]

[Page(model='gemini-2.5-flash-preview-04-17', prompt_version='2035c1e1', pkna='0', page='../input/pkna/pkna-0/pkna-0-029.jpg', takes=[Take(dialogue=[Bubble(character='Paperinik', text='Hai paura di affrontarmi? Fatti vedere!', probability=5), Bubble(character='Paperinik', text='Veramente, mi stai già guardando!', probability=5), Bubble(character='Donald Duck (inside robot)', text='Io sono questo edificio!', probability=5), Bubble(character='Donald Duck (inside robot)', text='Ma se proprio hai bisogno di un volto a cui rivolgerti... ecco!', probability=5), Bubble(character='Paperinik', text='Glom! Penso di averne viste abbastanza, per una sola notte!', probability=5), Bubble(character='Paperinik', text='Facciamoci coraggio!', probability=5), Bubble(character='Paperinik', text='Non so cosa tu sia, ma hai di fronte Paperinik!', probability=5), Bubble(character='Uno', text='Piacere! Io sono Uno!', probability=5), Bubble(character='Paperinik', text='Uno, eh? Dove hai lasciato gli altri?', p

In [23]:
# Normalize character names
import re
import copy
from collections import Counter

char_clusters = {
    'paperinik': ['.*paperinik.*', 'pk', '.*donald duck.*'],
}
char_clusters = {
    k: [re.compile(f"^{v}$") for v in vs]
    for k, vs in char_clusters.items()
}

def normalize_characters(page: Page) -> Page:
    page = copy.deepcopy(page)
    for take in page.takes:
        for bubble in take.dialogue:
            c = bubble.character.strip().lower()
            for k, vs in char_clusters.items():
                if any(re.match(pattern, c) for pattern in vs):
                    c = k
                    break
            bubble.character = c
    return page

norm_characters = [
    normalize_characters(page)
    for page in input_contents
]

character_counts = Counter(
    bubble.character
    for page in norm_characters
    for take in page.takes
    for bubble in take.dialogue
)

character_counts.most_common(10)

[('paperinik', 15065),
 ('uno', 8263),
 ('everett ducklair', 3565),
 ('lyla lay', 229),
 ('woman with red hair', 95),
 ('green creature', 89),
 ('green alien', 81),
 ('angus fangus', 80),
 ('lyla', 66),
 ('purple creature', 63)]

In [7]:
# Split by pkna and conversation
import pandas as pd

df = pd.DataFrame(
    [input_sort_key(f) for f in input_files],
    columns=['pkna', 'conv', 'page']
)

df.head()

,pkna,conv,page
0,0.0,29,0
1,0.0,29,1
2,0.0,29,2
3,0.0,29,3
4,0.0,29,4
